In [2]:
# INSTALL (RUN ONCE)
!pip install groq sentence-transformers faiss-cpu

# LOAD GROQ API KEY
import os
from google.colab import userdata

os.environ["api_key"] = userdata.get("api_key")

# IMPORTS
from groq import Groq

client = Groq(api_key=os.environ["api_key"])

# LLM FUNCTION
def ask_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

# STATE DEFINITION

def init_state(query):
    return {
        "query": query,
        "context": None,
        "reasoning": None,
        "answer": None,
        "valid": False,
        "retry_count": 0
    }

def retrieval_agent(state):
    query = state["query"].lower()

    # Simulated fintech knowledge base
    if "transaction" in query:
        context = "Transactions usually process within 24 hours."
    elif "fraud" in query:
        context = "Fraud detection flags suspicious activity automatically."
    elif "refund" in query:
        context = "Refunds are processed within 5-7 business days."
    else:
        context = "General FAQ: Contact support for more help."

    state["context"] = context
    return state

def reasoning_agent(state):
    prompt = f"""
    You are a fintech support assistant.

    Provide step-by-step reasoning.

    Query: {state['query']}
    Context: {state['context']}

    Answer clearly.
    """

    response = ask_llm(prompt)

    state["reasoning"] = response
    state["answer"] = response
    return state


def validation_agent(state):
    answer = state["answer"]

    # Simple validation logic
    if answer is None or len(answer) < 10:
        state["valid"] = False
    elif "I don't know" in answer:
        state["valid"] = False
    else:
        state["valid"] = True

    return state

# GRAPH WORKFLOW (LANGGRAPH STYLE)
def run_workflow(query, max_retries=2):

    state = init_state(query)

    while True:
        print("\n--- RETRIEVAL AGENT ---")
        state = retrieval_agent(state)

        print("Context:", state["context"])

        print("\n--- REASONING AGENT ---")
        state = reasoning_agent(state)

        print("Answer:", state["answer"])

        print("\n--- VALIDATION AGENT ---")
        state = validation_agent(state)

        print("Valid:", state["valid"])

        # Conditional edge (retry loop)
        if state["valid"]:
            print("\n FINAL ANSWER")
            return state["answer"]

        else:
            state["retry_count"] += 1

            if state["retry_count"] > max_retries:
                print("\n MAX RETRIES REACHED")
                return "Unable to provide a reliable answer."

            print("\n Retrying...\n")

# TESTING
queries = [
    "What happens if my transaction is delayed?",
    "How does fraud detection work?",
    "When will I get my refund?",
    "General help needed"
]

for q in queries:
    print("\n==============================")
    print("USER QUERY:", q)
    result = run_workflow(q)
    print("FINAL RESPONSE:", result)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.5 MB/s eta 0:00:00

USER QUERY: What happens if my transaction is delayed?

--- RETRIEVAL AGENT ---
Context: Transactions usually process within 24 hours.

--- REASONING AGENT ---
Answer: If your transaction is delayed, here's what you can expect:

1. **Initial Processing Time**: Transactions typically process within 24 hours. If your transaction hasn't been processed within this timeframe, it's considered delayed.
2. **Automated Review**: Our system will automatically review the transaction to identify any potential issues that may be causing the delay.
3. **Notification**: You will receive a notification from us via email or in-app message, informing you that your transaction has been delayed. This notification will provide you with an estimated time for when the issue is expected to be resolved.
4. **Manual Review**: If the automated review is 